In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!git clone -b add-librivox-demand-dataset https://github.com/hdgaribay/Audio_Enhancement_CNN.git
%cd Audio_Enhancement_CNN

In [ ]:
!pip install -q torch torchaudio soundfile soxr pesq pystoi pyyaml tqdm tensorboard pandas huggingface_hub

In [ ]:
import tarfile
import os

TAR_PATH   = "/content/drive/MyDrive/Audio_Enhancement_CNN/mixed_full.tar"
LOCAL_DATA = "/content/mixed_full"

if os.path.exists(LOCAL_DATA):
    count = len(os.listdir(f"{LOCAL_DATA}/train/clean"))
    print(f"Data already extracted — {count} train files ready")
else:
    print("Extracting mixed_full.tar to local storage...")
    print("This takes about 3-5 minutes...")
    with tarfile.open(TAR_PATH, "r") as tar:
        tar.extractall("/content")
    count = len(os.listdir(f"{LOCAL_DATA}/train/clean"))
    print(f"Extraction complete — {count} train files ready")

# symlink so scripts find it at data/mixed
os.makedirs("data", exist_ok=True)
if os.path.islink("data/mixed"):
    os.unlink("data/mixed")
os.symlink(LOCAL_DATA, "data/mixed")
print("data/mixed symlinked to local storage")

In [ ]:
!python scripts/make_manifest.py

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU --- go to Runtime > Change runtime type > L4 GPU'}")

In [ ]:
# Stage 2 — train audio denoiser
!python scripts/train_cnn.py --config config.yaml

In [ ]:
# Stage 1 — train IQ denoiser
!python scripts/train_iq.py --config config.yaml

In [ ]:
import shutil
import os

DRIVE_CHECKPOINTS = "/content/drive/MyDrive/Audio_Enhancement_CNN/checkpoints"
os.makedirs(DRIVE_CHECKPOINTS, exist_ok=True)
shutil.copytree("checkpoints/", DRIVE_CHECKPOINTS, dirs_exist_ok=True)
print(f"Checkpoints saved to: {DRIVE_CHECKPOINTS}")

In [ ]:
!python scripts/eval_models.py \
    --config config.yaml \
    --iq_checkpoint    checkpoints/iq_best.pt \
    --audio_checkpoint checkpoints/best.pt